In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import GoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [10]:
load_dotenv()

True

In [11]:
model = GoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [12]:
class LLMState(TypedDict):
    topic: str
    outline: str
    blog: str

In [18]:
def generate_outline(state: LLMState) -> LLMState:

    topic = state['topic']

    prompt = f'Generate a detailed outline for a blog on topic: {topic}'

    outline = model.invoke(prompt)

    state['outline'] = outline

    return state

In [14]:
def generate_blog(state: LLMState) -> LLMState:

    topic = state['topic']
    outline = state['outline']

    prompt = f'Generate a detailed blog for topic {topic} using following outline\n{outline}'

    blog = model.invoke(prompt)

    state['blog'] = blog

    return state

In [15]:
graph = StateGraph(LLMState)

graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_blog', generate_blog)

graph.add_edge(START, 'generate_outline')
graph.add_edge('generate_outline', 'generate_blog')
graph.add_edge('generate_blog', END)

workflow = graph.compile()

In [17]:
initial_state = { 'topic': "AI" }

final_state = workflow.invoke(initial_state)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
print(final_state['outline'])

In [ ]:
print(final_state['blog'])